# Train Geometry Segmenter

This notebook trains a two-class YOLO segmentation model for table and net geometry from datasets exported by `tools/export_yolo/export_geometry_dataset.py`.

Expected input after export:

```txt
data/exports/geometry_yolo_seg/
  dataset.yaml
  images/train/*.jpg
  images/val/*.jpg
  labels/train/*.txt
  labels/val/*.txt
```

This is useful as a table+net pipeline smoke test even with a small dataset. The net class is a thin segmentation band derived from the labeled net polyline, so expect early runs to overfit but still reveal whether the model can find the net stripe.

## 1. Runtime Check

In Colab, choose `Runtime > Change runtime type > GPU` before training.

In [ ]:
import sys
import platform

print("python", sys.version)
print("platform", platform.platform())

try:
    import torch
    print("torch", torch.__version__)
    print("cuda available", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu", torch.cuda.get_device_name(0))
except ImportError:
    print("torch not installed yet")

python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
platform Linux-6.6.122+-x86_64-with-glibc2.35
torch 2.11.0+cu128
cuda available True
gpu Tesla T4


## 2. Install Training Dependencies

In [ ]:
%pip install -q ultralytics

## 3. Provide Dataset

Run the next cell in Colab and choose `geometry_yolo_seg.zip` when the picker appears. If you see a greyed-out old picker, ignore that old output and rerun the cell with the play button to create a fresh picker.

If `geometry_yolo_seg.zip` already exists in `/content`, the cell will reuse it without opening the picker.

Locally, create it after export with:

```bash
cd data/exports
zip -r geometry_yolo_seg.zip geometry_yolo_seg
```

In [ ]:
from IPython.display import clear_output
clear_output(wait=False)
print("Upload output reset. Now run the next cell.")

In [ ]:
from pathlib import Path
import shutil
import zipfile
from IPython.display import clear_output

clear_output(wait=False)

DATASET_ZIP = "geometry_yolo_seg.zip"
extract_dir = Path("/content/geometry_yolo_seg")

def find_existing_zip():
    candidates = [Path(DATASET_ZIP), Path("/content") / DATASET_ZIP]
    candidates.extend(Path("/content").glob("geometry_yolo_seg*.zip"))
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

zip_path = find_existing_zip()
if zip_path is None:
    try:
        from google.colab import files
    except ModuleNotFoundError as exc:
        raise RuntimeError("This upload cell must be run in Google Colab, or geometry_yolo_seg.zip must already exist in the current directory.") from exc

    print(f"Choose {DATASET_ZIP}. If the button is greyed out, ignore the old output and rerun this cell with the play button.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded. Rerun this cell and choose geometry_yolo_seg.zip.")

    if DATASET_ZIP in uploaded:
        zip_path = Path(DATASET_ZIP)
    else:
        zip_names = [name for name in uploaded if name.endswith(".zip")]
        if len(zip_names) != 1:
            raise FileNotFoundError(f"Expected {DATASET_ZIP} or exactly one .zip file, got: {list(uploaded)}")
        zip_path = Path(zip_names[0])
        print(f"Using uploaded zip: {zip_path}")

if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True, exist_ok=True)

print("using zip:", zip_path.resolve())
print("zip size MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))

with zipfile.ZipFile(zip_path) as archive:
    members = archive.infolist()
    total = len(members)
    print(f"extracting {total} files...")
    for index, member in enumerate(members, start=1):
        archive.extract(member, extract_dir)
        if index == total or index % 10 == 0:
            clear_output(wait=True)
            print("using zip:", zip_path.resolve())
            print("zip size MB:", round(zip_path.stat().st_size / 1024 / 1024, 2))
            print(f"extracting files: {index}/{total}")

matches = sorted(extract_dir.rglob("dataset.yaml"))
if not matches:
    found = sorted(str(path.relative_to(extract_dir)) for path in extract_dir.rglob("*") if path.is_file())[:20]
    raise FileNotFoundError(f"No dataset.yaml found after extracting {zip_path}. First extracted files: {found}")

DATASET_YAML = str(matches[0])
dataset_root = Path(DATASET_YAML).parent.resolve()
Path(DATASET_YAML).write_text(
    f"path: {dataset_root.as_posix()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: table\n"
    "  1: net\n",
    encoding="utf-8",
)

print("dataset yaml:", DATASET_YAML)
print("dataset root:", dataset_root)

## 4. Inspect Dataset Size

In [ ]:
dataset_root = Path(DATASET_YAML).parent
total_images = 0
total_labels = 0
for split in ("train", "val"):
    image_count = len(list((dataset_root / "images" / split).glob("*")))
    label_count = len(list((dataset_root / "labels" / split).glob("*.txt")))
    total_images += image_count
    total_labels += label_count
    print(split, "images", image_count, "labels", label_count)

if total_images == 0 or total_labels == 0:
    raise RuntimeError(f"Dataset appears empty at {dataset_root}. Re-upload data/exports/geometry_yolo_seg.zip and rerun the dataset cell.")

## 5. Train Baseline

Start with the nano segmentation model. This run is intentionally conservative for free Colab RAM; once it works, increase `imgsz` or `batch` gradually.

In [ ]:
from ultralytics import YOLO

dataset_root = Path(DATASET_YAML).parent.resolve()
Path(DATASET_YAML).write_text(
    f"path: {dataset_root.as_posix()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "names:\n"
    "  0: table\n"
    "  1: net\n",
    encoding="utf-8",
)
print(Path(DATASET_YAML).read_text())

model = YOLO("yolo26n-seg.pt")
results = model.train(
    data=DATASET_YAML,
    epochs=30,
    imgsz=640,
    batch=2,
    workers=2,
    cache=False,
    patience=10,
    project="/content/runs/ar_ping_pong",
    name="geometry_yolo26n_seg_img640",
)

## 6. Validate And Predict

In [ ]:
best = "/content/runs/ar_ping_pong/geometry_yolo26n_seg_img640/weights/best.pt"
model = YOLO(best)
metrics = model.val(data=DATASET_YAML, imgsz=640)
print(metrics)

prediction_source = dataset_root / "images" / "val"
if not any(prediction_source.glob("*")):
    prediction_source = dataset_root / "images" / "train"

predictions = model.predict(
    source=str(prediction_source),
    imgsz=640,
    conf=0.1,
    save=True,
    project="/content/runs/ar_ping_pong",
    name="geometry_predictions",
)
print("prediction images:", "/content/runs/ar_ping_pong/geometry_predictions")

## 7. Next Comparison

Once the tiny run works, try more table/net labels, then rerun with `epochs=100`. A video-level validation split will only be meaningful after several different videos are labeled.